In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [1]:
dataset_path = "Skin_Conditions"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),  # MobileNetV2 expects 224x224
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


NameError: name 'ImageDataGenerator' is not defined

In [5]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze the base model layers
base_model.trainable = False

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(6, activation='softmax')(x)  # 6 classes

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 95s 10us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
model.save("skin_condition_model.h5")


In [4]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze base layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(6, activation='softmax')(x)  # 6 skin classes

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [8]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)


Epoch 1/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 211s 3s/step - accuracy: 0.5005 - loss: 1.3408 - val_accuracy: 0.7300 - val_loss: 0.8232
Epoch 2/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.6839 - loss: 0.8942 - val_accuracy: 0.7764 - val_loss: 0.6575
Epoch 3/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.7193 - loss: 0.7430 - val_accuracy: 0.7911 - val_loss: 0.5997
Epoch 4/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.7427 - loss: 0.6963 - val_accuracy: 0.8059 - val_loss: 0.5724
Epoch 5/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.7849 - loss: 0.5960 - val_accuracy: 0.8101 - val_loss: 0.5244
Epoch 6/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.7885 - loss: 0.5710 - val_accuracy: 0.8354 - val_loss: 0.4938
Epoch 7/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 0.8073 - loss: 0.5543 - val_accuracy: 0.8376 - val_loss: 0.4914
Epoch 8/10
60/60 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.8031 - loss: 0.5040 - val_accuracy: 0.8080 - v

In [6]:
base_model.trainable = True
for layer in base_model.layers[:-30]:  # freeze first layers, train last 30 layers
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # lower learning rate
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_finetune = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

Epoch 1/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 228s 3s/step - accuracy: 0.1906 - loss: 2.0629 - val_accuracy: 0.2384 - val_loss: 2.1105
Epoch 2/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 175s 3s/step - accuracy: 0.3203 - loss: 1.7121 - val_accuracy: 0.2827 - val_loss: 1.8628
Epoch 3/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 174s 3s/step - accuracy: 0.4214 - loss: 1.4848 - val_accuracy: 0.3270 - val_loss: 1.7147
Epoch 4/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 200s 3s/step - accuracy: 0.5042 - loss: 1.2978 - val_accuracy: 0.3755 - val_loss: 1.5183
Epoch 5/5
60/60 ━━━━━━━━━━━━━━━━━━━━ 180s 3s/step - accuracy: 0.5609 - loss: 1.2031 - val_accuracy: 0.4473 - val_loss: 1.3937


In [7]:
model.save("skin_condition_model.h5")


In [8]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the trained model
model = load_model("skin_condition_model.h5")
print("Model loaded ✅")


Model loaded ✅


In [9]:
img_path = "test_skin.jpg"  # replace with your image path

# Load image & resize to 224x224 (MobileNetV2 input size)
img = image.load_img(img_path, target_size=(224,224))

# Convert to array & normalize
img_array = image.img_to_array(img) / 255.0

# Add batch dimension (Keras expects shape: [1, 224,224,3])
img_array = np.expand_dims(img_array, axis=0)


In [10]:
# Get predictions
preds = model.predict(img_array)  # returns probabilities for each class

# Get class with highest probability
class_index = np.argmax(preds[0])

# Map index to actual label
labels = list(train_generator.class_indices.keys())  # automatically gets your 6 class names
predicted_label = labels[class_index]

# Probability of predicted class
probability = preds[0][class_index]

print(f"Predicted skin condition: {predicted_label}")
print(f"Prediction confidence: {probability*100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Predicted skin condition: Eczema
Prediction confidence: 27.47%
